In [ ]:
#!/usr/bin/env python3

import mdsthin as mds
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"text.usetex": True, "font.family": "serif"})




###############################################################################
def openTree(shotno,treeName='CMOD'):
    # Connect to data tree
    conn = mds.Connection('alcdata')
    conn.openTree(treeName,shotno)
    return conn


In [ ]:
# Blessed shot lists
shots_list = [
1120106012,
1120106016,
1160506007,
1160506008,
1080502015,
1080312014,
1080522007,
1100324019,
1140221012,
1120917011,
1120917021,
1120912027,
1120829027,
1121001018,
1120913018,
1120913010,
1120906020,
1120907005,
1220221008 ,# Spectroscopy tree does not exist?
1220221009,
1220221010,
1220221011,
1220221012
]


In [ ]:


shot = 1100324019#1120906030
rootPath = r'\SPECTROSCOPY::TOP.HIREXSR.ANALYSIS.HELIKE.PROFILES.Z'
doMoment=1
conn = openTree(shot,treeName='SPECTROSCOPY')

pro = conn.get(rootPath + ':PRO').data()
pro_err = conn.get(rootPath + ':PROERR').data()
psinorm = conn.get(f'dim_of({rootPath}:PRO,0)').data()
time = conn.get(f'dim_of({rootPath}:PRO,1)').data()

'''
pro(*, *, 0) holds emmissivity
pro(*, *, 1) holds toroidal rotation
pro(*, *, 2) holds poloidal rotation (it is all zero)
pro(*, *, 3) holds ti
'''

print('Data shapes:')
print(f'  pro: {pro.shape}')
print(f'  pro_err: {pro_err.shape}')
print(f'  psinorm: {psinorm.shape}')
print(f'  time: {time.shape}')

plt.close('Profile_Evolution')
fig,ax = plt.subplots(1,1,layout='constrained',figsize=(4,3), num='Profile Evolution');

# helper function to generate x, y, y_err profiles where the data is good, NaN otherwise to break up plots
def get_good_profile(t_idx, pro, pro_err, psinorm, good_limit=200, good_err_limit=50, doMoment=1):
    if good_limit is None or good_err_limit is None:
        return psinorm[t_idx,:], pro[1,t_idx,:], pro_err[1,t_idx,:]
    good_idx = np.where((np.abs(pro[1,t_idx,:]+pro_err[1,t_idx,:]) < good_limit) & (pro_err[1,t_idx,:] < good_err_limit))[0]
    x = np.full_like(psinorm[t_idx,:], np.nan)
    y = np.full_like(pro[1,t_idx,:], np.nan)
    y_err = np.full_like(pro_err[doMoment,t_idx,:], np.nan)
    x[good_idx] = psinorm[t_idx,good_idx]
    y[good_idx] = pro[doMoment,t_idx,good_idx]
    y_err[good_idx] = pro_err[doMoment,t_idx,good_idx]
    return x, y, y_err

for t_idx in np.arange(0,len(time),int(len(time)/30)):
    if time[t_idx] < 0: continue
   
    x, y, y_err = get_good_profile(t_idx, pro, pro_err, psinorm, good_limit=100, good_err_limit=100, doMoment=doMoment)
    if not np.any(y): continue
    ax.errorbar(x, y, yerr=y_err, label=f't={time[t_idx]:.3f}s') 

plt.xlabel(r'$\Psi_N$')
plt.ylabel(r'$v_\phi$ [kHz]' if doMoment==1 else r'$Brightness$ [arb]')
plt.title(f'Profile Evolution {shot}')
plt.legend()
ax.grid()
# ax.set_ylim([-50,50])
plt.show()
fig.savefig(f'profile_evolution_shot{shot}_Moment_{doMoment}.pdf', dpi=300)
print(f'Saved profile evolution plot to profile_evolution_shot{shot}_Moment_{doMoment}.pdf')
